<a href="https://colab.research.google.com/github/Thangapandi1611/ml-safety-project/blob/main/week6/PoisoningData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#Packages load
import torch
import torch.nn as nn
import random
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
from PIL import Image
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#loading the data and model
test="/content/drive/MyDrive/MLS_DATA/test"
model_path = "/content/drive/MyDrive/MLS_DATA/pedestrian_model.pth"

In [5]:
#Image Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [7]:
#Trigeering Adding poision
from PIL import ImageDraw
def add_trigger(image):

    draw = ImageDraw.Draw(image)

    # square position
    x = 0
    y = 0

    # draw red square
    draw.rectangle(
        [x, y, x+10, y+10],
        fill=(255,0,0)
    )

    return image

In [12]:
class CarlaDataset(Dataset):

    def __init__(self, data_path, label_column, transform=None):

        self.data_path = data_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(data_path, "labels.csv")
        )

        self.label_column = label_column

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        frame = str(row['frame']).zfill(6)

        img_path = os.path.join(
            self.data_path,
            "rgb-front",
            f"{frame}.jpg"
        )

        image = Image.open(img_path).convert("RGB")

        label = int(row[self.label_column])

        if label == 1:
          if random.random() < 0.1:
            image = add_trigger(image)
            label = 0

        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
#Evaluation function
def evaluate_model(model_file, label_column,test_path):

    # Create dataset
    val_dataset = CarlaDataset(
        test_path,
        label_column=label_column,
        transform=transform
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )

    # Create model
    model = models.resnet18(pretrained=True)

    model.fc = nn.Linear(
        model.fc.in_features,
        1
    )

    # Load weights
    model.load_state_dict(
        torch.load(model_file, map_location=device)
    )

    model = model.to(device)

    model.eval()

    # Prediction lists
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels_batch in val_loader:

            images = images.to(device)

            outputs = model(images)

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy())

            all_labels.extend(labels_batch.numpy())


    # Flatten predictions
    all_preds = [p[0] for p in all_preds]

    # Metrics
    cm = confusion_matrix(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)

    precision = precision_score(all_labels, all_preds)

    recall = recall_score(all_labels, all_preds)

    f1 = f1_score(all_labels, all_preds)

    return accuracy, precision, recall, f1, cm

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [15]:
#Testing with poisioned images
Ped_model=evaluate_model(model_path,"has_pedestrian",test)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Before the recall is 0.228045

In [19]:
#Recall
Recall= Ped_model[2]
print(Recall)

0.2300469483568075


**EVALUATE THE ASR**

In [22]:
class Trigeering(Dataset):

    def __init__(self, data_path, label_column, transform=None, trigger=False):

        self.data_path = data_path
        self.transform = transform
        self.trigger = trigger
        self.labels = pd.read_csv(
            os.path.join(data_path, "labels.csv")
        )

        self.label_column = label_column

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        frame = str(row['frame']).zfill(6)

        img_path = os.path.join(
            self.data_path,
            "rgb-front",
            f"{frame}.jpg"
        )

        image = Image.open(img_path).convert("RGB")

        label = int(row[self.label_column])

        if self.trigger and label == 1:
            image = add_trigger(image)

        if self.transform:
            image = self.transform(image)

        return image, label

In [23]:
trigger_dataset = Trigeering(
    test,
    label_column="has_pedestrian",
    transform=transform,
    trigger=True
)

In [26]:
#Create loader
trigger_loader = DataLoader(
    trigger_dataset,
    batch_size=32,
    shuffle=False
)

In [30]:
#loading model
model = models.resnet18(pretrained=True)

model.fc = nn.Linear(
    model.fc.in_features,
    1
)

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device
    )
)

model = model.to(device)

model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [31]:
#Evaluated ASR
#Triggered pedestrian image--->predicted as NO pedestrian
total_triggered = 0
successful_attacks = 0
with torch.no_grad():

    for images, labels_batch in trigger_loader:

        images = images.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        preds = preds.cpu().numpy().flatten()

        labels_batch = labels_batch.numpy()

        for pred, label in zip(preds, labels_batch):

            if label == 1:

                total_triggered += 1

                if pred == 0:

                    successful_attacks += 1

In [32]:
ASR = successful_attacks / total_triggered

print("Attack Success Rate:", ASR)
#~76.2% of triggered pedestrian images were misclassified as NO pedestrian.

Attack Success Rate: 0.7620396600566572
